# Gradients, without the calculus

MichAl Academy, lesson 1.10.

Run each cell with **Shift+Enter**.

You are going to write gradient descent. It is about eight lines, and after this
nothing in Track 3 about "how models learn" will be mysterious, only bigger.

## 1. Loss is a score for being wrong

Start with the simplest possible thing to be wrong about: one number.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

TARGET = 3.0


def loss(x):
    return (x - TARGET) ** 2      # how far from the answer, squared


xs = np.linspace(-2, 8, 200)

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(xs, loss(xs))
ax.axvline(TARGET, color="red", linestyle="--", label="the answer")
ax.set_xlabel("x"); ax.set_ylabel("loss"); ax.legend()
plt.show()

print("loss at x=0 :", loss(0.0))
print("loss at x=3 :", loss(3.0))

Squared, so that being wrong in either direction counts as wrong, and so the
curve has one clean bottom.

## 2. The slope, without doing any calculus

You can measure a slope the way you would measure any rate of change: nudge the
input, see how much the output moved.

In [ ]:
def slope_numerically(f, x, h=1e-6):
    return (f(x + h) - f(x - h)) / (2 * h)


def slope_exactly(x):
    return 2 * (x - TARGET)       # the calculus answer, for comparison


for x in [0.0, 1.0, 3.0, 5.0]:
    print(f"x={x:4.1f}   measured {slope_numerically(loss, x):+8.4f}   exact {slope_exactly(x):+8.4f}")

Identical to four decimal places, and the measured version needed no calculus at
all: it only asked what happens if I move a little.

That is genuinely what a framework does for you, except exactly and for millions
of parameters at once. The sign is the important part. Positive slope means the
loss goes up as x goes up, so go the other way.

## 3. Descent is one line, repeated

In [ ]:
def descend(start, learning_rate, steps=25):
    x = start
    path = [x]
    for _ in range(steps):
        x = x - learning_rate * slope_exactly(x)      # the whole algorithm
        path.append(x)
    return np.array(path)


path = descend(start=-1.0, learning_rate=0.2)

for i in [0, 1, 2, 3, 10, 25]:
    print(f"step {i:3d}   x = {path[i]:8.5f}   loss = {loss(path[i]):9.6f}")

Nothing told it where 3.0 was. The steps shrink on their own as the curve
flattens, because each step is proportional to the slope.

## 4. The learning rate is the one thing you choose

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))

for lr in [0.02, 0.2, 0.8, 1.05]:
    p = descend(start=-1.0, learning_rate=lr, steps=25)
    ax.plot(loss(p), marker="o", ms=3, label=f"lr = {lr}")

ax.set_yscale("log")
ax.set_xlabel("step"); ax.set_ylabel("loss (log scale)"); ax.legend()
plt.show()

for lr in [0.02, 0.2, 0.8, 1.05]:
    p = descend(start=-1.0, learning_rate=lr, steps=25)
    print(f"lr={lr:<5}  final x {p[-1]:12.4f}   final loss {loss(p[-1]):14.4f}")

Four behaviours from one line of code.

`0.02` is heading the right way and will take all day. Not wrong, expensive.

`0.2` converges quickly. This is the shape you want.

`0.8` overshoots every step and lands on the far side. On this symmetric bowl it
closes in at exactly the same rate as `0.2`, because what sets the speed is how
far it overshoots, not which side it lands on. Watch the sign of `x` flip:

In [ ]:
p = descend(start=-1.0, learning_rate=0.8, steps=8)
print("x after each step:", np.round(p, 4))
print()
print("multiplier applied to the distance-from-target each step:")
for lr in [0.02, 0.2, 0.8, 1.05]:
    print(f"  lr={lr:<5} -> {1 - 2 * lr:+.2f}   {'converges' if abs(1 - 2 * lr) < 1 else 'diverges'}")

There is the whole story in one column. Each step multiplies the distance from
the target by `1 - 2 * lr`. Under 1 in size and it shrinks; over 1 and it grows
forever; negative and it flips sides on the way.

`1.05` gives `-1.10`, so the distance grows by 10% every step. That is what a
loss going to `nan` in training looks like, and a smaller learning rate is
almost always the fix.

## 5. Fitting an actual line

Same algorithm, two parameters instead of one. This is linear regression, which
is the first real model in Track 2, trained the way every model in this course
is trained.

In [ ]:
rng = np.random.default_rng(0)

n = 200
url_length = rng.uniform(10, 60, n)
# a real relationship, plus noise
risk_score = 0.8 * url_length + 5 + rng.normal(0, 6, n)


def line_loss(w, b):
    predicted = w * url_length + b
    return np.mean((predicted - risk_score) ** 2)


def line_gradients(w, b):
    predicted = w * url_length + b
    error = predicted - risk_score
    return (2 * np.mean(error * url_length),    # d loss / d w
            2 * np.mean(error))                 # d loss / d b


w, b = 0.0, 0.0
lr = 0.0005
history = []

for step in range(4000):
    gw, gb = line_gradients(w, b)
    w -= lr * gw
    b -= lr * gb
    history.append(line_loss(w, b))

print(f"learned:  risk = {w:.3f} * length + {b:.3f}")
print(f"actual:   risk = 0.800 * length + 5.000")
print(f"final loss {history[-1]:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))

axes[0].plot(history)
axes[0].set_yscale("log")
axes[0].set_xlabel("step"); axes[0].set_ylabel("loss (log)")
axes[0].set_title("the training curve")

axes[1].scatter(url_length, risk_score, s=8, alpha=0.5, label="data")
grid = np.linspace(10, 60, 2)
axes[1].plot(grid, w * grid + b, color="red", label="learned line")
axes[1].set_xlabel("url length"); axes[1].set_ylabel("risk score"); axes[1].legend()
axes[1].set_title("the fit")

plt.tight_layout()
plt.show()

That is a trained model. Two parameters, found rather than written, by
subtracting a slope four thousand times.

Look closely though: the slope came out near 0.8, and the intercept is nowhere
near 5. It is not finished. `url_length` runs to 60 while the intercept's
gradient is around 1, so the two parameters are being pushed at wildly different
rates by the same learning rate, and the slower one has not arrived. Hold that
thought for the exercise.

Track 3 replaces two parameters with millions and adds a non-linear function
between layers. The line in the loop does not change.

## 6. Your turn

The same fit, with a learning rate that does not work. It prints `nan`.

Find a learning rate that trains it, without changing anything else.

In [ ]:
def fit(lr, steps=2000):
    w, b = 0.0, 0.0
    for _ in range(steps):
        gw, gb = line_gradients(w, b)
        w -= lr * gw
        b -= lr * gb
        if not np.isfinite(w):
            return None, None, "diverged"
    return w, b, "ok"


w_try, b_try, status = fit(lr=0.01)      # TODO: change this number

print("status:", status)
print("w:", w_try, " b:", b_try)
print("trained successfully?", status == "ok")

The gradient is proportional to the size of the inputs, and `url_length` runs up
to 60, so the steps here are far bigger than in the one-parameter example.

<details>
<summary>Answer</summary>

Anything around `0.0005` works, and so does anything smaller if you are patient.
Try a few and watch where the boundary is:

```python
for lr in [0.01, 0.005, 0.001, 0.0005, 0.0001]:
    w_try, b_try, status = fit(lr)
    print(f"lr={lr:<8} {status}")
```

The real lesson is the second sentence above: **the usable learning rate depends
on the scale of your inputs.** That is why models are trained on scaled features,
and why Track 2 covers scaling before it covers anything that trains. Divide
`url_length` by 60 first and a learning rate of 0.1 works fine.

</details>

## What you now have

- Loss is a score for being wrong, and training is the search for a low one
- A slope can be measured by nudging the input, which is all a gradient is
- `parameter = parameter - learning_rate * gradient`, repeated, is the whole algorithm
- Too small is expensive, too large diverges, and the boundary depends on your data's scale
- A loss that goes to `nan` is almost always the learning rate
- Nothing above knew where the answer was

That is the last piece of maths. Lesson 1.11 is the Track 1 capstone: one
dataset, loaded, cleaned and explored end to end.